## Dataset Alignment and Patient-Level Labeling

This notebook aligns the clinical and imaging datasets and converts the clinical dataset from breast-level (Left/Right) to patient-level.

Key objectives:
- Match clinical and image datasets using Patient IDs
- Remove unmatched data
- Convert L/R rows into patient-level labels
- Generate a clean dataset for Baseline A, B, and C

In [1]:
import os
import shutil
import pandas as pd
from tqdm import tqdm

## Load Clinical Dataset

The dataset contains multiple rows per patient (Left and Right breasts).

In [ ]:
clinical_path = "cleaned_clinical_dataset.csv"

df = pd.read_csv(clinical_path)

print("Original Shape:", df.shape)
print("Unique Patients:", df["ID"].nunique())

df.head()

## Extract Unique Patient IDs

These IDs will be used to match with the image dataset.

In [ ]:
clinical_ids = set(df["ID"].unique())

print("Total Clinical Patients:", len(clinical_ids))

## Load Image Dataset

Each folder in CMMD corresponds to a patient.

In [ ]:
CMMD_PATH = "CMMD"
OUTPUT_PATH = "CMMD_Aligned"

os.makedirs(OUTPUT_PATH, exist_ok=True)

image_folders = os.listdir(CMMD_PATH)

print("Total Image Folders:", len(image_folders))

## Align Image Dataset

Copy only those patient folders that exist in the clinical dataset.

In [ ]:
matched_patients = []

for folder in tqdm(image_folders):
    if folder in clinical_ids:
        src = os.path.join(CMMD_PATH, folder)
        dst = os.path.join(OUTPUT_PATH, folder)

        shutil.copytree(src, dst, dirs_exist_ok=True)
        matched_patients.append(folder)

print("Matched Patients:", len(matched_patients))

## Filter Clinical Dataset

Keep only patients that have corresponding image data.

In [ ]:
aligned_df = df[df["ID"].isin(matched_patients)].copy()

print("Filtered Shape:", aligned_df.shape)
print("Filtered Patients:", aligned_df["ID"].nunique())

## Convert to Patient-Level Dataset

Each patient may have multiple rows (Left/Right).

We convert to one row per patient using:
- target → max (if any malignant → patient = malignant)
- other features → mean (or first, depending on feature type)

In [ ]:
# Separate target
patient_df = aligned_df.groupby("ID").agg({
    "target": "max"
})

# Handle other features automatically
for col in aligned_df.columns:
    if col not in ["ID", "target"]:
        # Use mean for numerical features
        if aligned_df[col].dtype != "object":
            patient_df[col] = aligned_df.groupby("ID")[col].mean()
        else:
            # Use first value for categorical
            patient_df[col] = aligned_df.groupby("ID")[col].first()

patient_df = patient_df.reset_index()

print("Patient-Level Shape:", patient_df.shape)
print("Total Patients:", patient_df["ID"].nunique())

patient_df.head()

## Save Final Aligned Dataset

This dataset is now:
- Patient-level
- Fully aligned with image data
- Ready for all baselines

In [ ]:
patient_df.to_csv("aligned_clinical_dataset.csv", index=False)

print("Saved: aligned_clinical_dataset.csv")